# LSTM Retraining with Imputation

This notebook retrains the LSTM from scratch on imputed training data.
Only Black patients receive imputation. White patients are untouched.

## Three conditions compared:
- **Baseline** original unmodified data
- **Variant A** Black missing slots filled to match White observation rate
- **Variant B** Black missing slots fully filled every hour (theoretical ceiling)

Imputation is applied to BOTH training AND test data consistently.
The model learns from imputed data and is evaluated on imputed data.
This is the correct experimental design for testing whether imputation
can reduce bias at the training level.

---
## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
from pathlib import Path
from sklearn.metrics import (
    average_precision_score, roc_auc_score, precision_recall_curve
)
from sklearn.model_selection import StratifiedKFold
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})

OUT_DIR  = Path('/home/ino/thesis_outputs')
IMP_DIR  = OUT_DIR / 'imputation_retrain'
IMP_DIR.mkdir(exist_ok=True)

VITAL_NAMES  = ['heart_rate', 'resp_rate', 'spo2', 'temperature', 'map']
KEEP_RACES   = ['White', 'Black', 'Hispanic', 'Asian']
N_FOLDS      = 5
RANDOM_STATE = 42
N_BOOTSTRAP  = 2000

# Training hyperparameters (identical to original thesis_cv_full)
N_EPOCHS   = 50
BATCH_SIZE = 1024
LR         = 5e-4

COLORS = {
    'Baseline':  '#7f8c8d',
    'Variant A': '#2980b9',
    'Variant B': '#27ae60',
}
RACE_COLORS = {
    'White': '#4878CF', 'Black': '#D65F5F',
    'Hispanic': '#6ACC65', 'Asian': '#B47CC7'
}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
n_gpus = torch.cuda.device_count()
print(f'Device: {device}  |  GPUs: {n_gpus}')
print(f'Output: {IMP_DIR}')

---
## 2. Load & align data

In [ ]:
# Identical alignment to all previous notebooks
cv_stay_ids = np.load(OUT_DIR / 'cv_stay_ids.npy')
print(f'CV stay_ids: {len(cv_stay_ids):,}')

cohort_full = pd.read_csv(OUT_DIR / 'cohort.csv')
cohort_full = cohort_full.set_index('stay_id')
cohort_cv   = cohort_full.loc[cv_stay_ids].reset_index()

cohort_full_reset = pd.read_csv(OUT_DIR / 'cohort.csv').reset_index(drop=True)
stay_id_to_ts_idx = {sid: i for i, sid in
                     enumerate(cohort_full_reset['stay_id'].values)}

X_ts_full  = np.load(OUT_DIR / 'X_ts.npy')
M_ts_full  = np.load(OUT_DIR / 'M_ts.npy')
ts_indices = np.array([stay_id_to_ts_idx[sid] for sid in cv_stay_ids])
X_ts_cv    = X_ts_full[ts_indices]
M_ts_cv    = M_ts_full[ts_indices]

keep_mask = cohort_cv['race_group'].isin(KEEP_RACES).values
cohort    = cohort_cv[keep_mask].reset_index(drop=True)
kept_idx  = np.where(keep_mask)[0]

X_ts    = X_ts_cv[kept_idx]   # (N, 24, 5)
M_ts    = M_ts_cv[kept_idx]   # (N, 24, 5)

y        = cohort['hospital_expire_flag'].values.astype(np.int32)
demo     = cohort[['race_group', 'gender', 'age_group']].reset_index(drop=True)
race_arr = demo['race_group'].values

print(f'Cohort: {len(cohort):,}  |  Mortality: {y.mean():.1%}')
print(cohort['race_group'].value_counts().to_string())

# Load baseline OOF probs (already trained, no need to retrain baseline)
oof_baseline = np.load(OUT_DIR / 'oof_probs_LSTM.npy')[kept_idx]
bl_overall   = average_precision_score(y, oof_baseline)
bl_black     = average_precision_score(
    y[race_arr=='Black'], oof_baseline[race_arr=='Black'])
bl_white     = average_precision_score(
    y[race_arr=='White'], oof_baseline[race_arr=='White'])

print(f'\nBaseline LSTM AUPRC (sanity check):')
print(f'  Overall: {bl_overall:.4f}')
print(f'  Black:   {bl_black:.4f}')
print(f'  White:   {bl_white:.4f}')
print(f'  Gap:     {bl_white-bl_black:+.4f}')

---
## 3. Fold splits

In [ ]:
strat_label = (demo['race_group'].astype(str) + '_' +
               cohort['hospital_expire_flag'].astype(str)).values
skf         = StratifiedKFold(n_splits=N_FOLDS, shuffle=True,
                              random_state=RANDOM_STATE)
fold_splits  = list(skf.split(np.zeros(len(cohort)), strat_label))
print(f'Fold test sizes: {[len(t) for _,t in fold_splits]}')

---
## 4. Imputation functions

Variant A: fill to match White observation rate


Variant B: fill all missing slots


In [ ]:
def compute_fold_stats(train_idx):
    """
    Compute per-vital per-hour statistics from training fold.
    Returns:
        black_median[v][h] = median value for Black training patients
        black_rate[v][h]   = observation rate for Black training patients
        white_rate[v][h]   = observation rate for White training patients
    """
    black_tr  = train_idx[race_arr[train_idx] == 'Black']
    white_tr  = train_idx[race_arr[train_idx] == 'White']

    X_bl = X_ts[black_tr]   # (N_black, 24, 5)
    M_bl = M_ts[black_tr]
    M_wh = M_ts[white_tr]

    black_median = {}
    black_rate   = {}
    white_rate   = {}

    for v in range(len(VITAL_NAMES)):
        black_median[v] = {}
        black_rate[v]   = {}
        white_rate[v]   = {}
        for h in range(24):
            obs = M_bl[:, h, v] == 1
            black_median[v][h] = float(np.median(X_bl[:, h, v][obs])) \
                                 if obs.any() else float(np.median(X_bl[:, h, v]))
            black_rate[v][h]   = float(M_bl[:, h, v].mean())
            white_rate[v][h]   = float(M_wh[:, h, v].mean())

    return black_median, black_rate, white_rate


def apply_variant_a(X, M, race_te, black_median, black_rate,
                    white_rate, rng):
    """
    Variant A: fill Black missing slots to match White observation rate.
    Only fills slots where White rate > Black rate.
    Fill probability = (white_rate - black_rate) / (1 - black_rate)
    This gives the correct expected rate after imputation.
    """
    X_out = X.copy()
    M_out = M.copy()
    black_loc = np.where(race_te == 'Black')[0]

    for li in black_loc:
        for v in range(len(VITAL_NAMES)):
            for h in range(24):
                if M_out[li, h, v] == 0:   # only fill missing slots
                    br = black_rate[v][h]
                    wr = white_rate[v][h]
                    if wr > br:            # White rate is higher
                        # Fill with probability that brings Black up to White rate
                        fill_prob = (wr - br) / (1.0 - br + 1e-8)
                        if rng.random() < fill_prob:
                            X_out[li, h, v] = black_median[v][h]
                            M_out[li, h, v] = 1.0
    return X_out, M_out


def apply_variant_b(X, M, race_te, black_median):
    """
    Variant B: fill ALL Black missing slots with Black median.
    Every Black patient ends up with a complete observation matrix.
    """
    X_out = X.copy()
    M_out = M.copy()
    black_loc = np.where(race_te == 'Black')[0]

    for li in black_loc:
        for v in range(len(VITAL_NAMES)):
            for h in range(24):
                if M_out[li, h, v] == 0:
                    X_out[li, h, v] = black_median[v][h]
                    M_out[li, h, v] = 1.0
    return X_out, M_out


print('Imputation functions defined.')

# Quick sanity check on observation rates
# Use fold 0 train to show the before/after rates for Variant A
bmed, brate, wrate = compute_fold_stats(fold_splits[0][0])
print('\nObservation rates — fold 0 training data:')
print(f'{"Vital":<14}  {"Black rate":>12}  {"White rate":>12}  {"Gap":>8}')
for v, vname in enumerate(VITAL_NAMES):
    br = np.mean([brate[v][h] for h in range(24)])
    wr = np.mean([wrate[v][h] for h in range(24)])
    print(f'{vname:<14}  {br:>12.3f}  {wr:>12.3f}  {wr-br:>8.3f}')

---
## 5. LSTM model & training

In [ ]:
class MortalityLSTM(nn.Module):
    """Identical to thesis_cv_full.ipynb — 3-layer BiLSTM"""
    def __init__(self, input_size=10, hidden_size=128,
                 num_layers=3, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size, hidden_size=hidden_size,
            num_layers=num_layers, batch_first=True,
            dropout=dropout, bidirectional=True
        )
        self.layer_norm = nn.LayerNorm(hidden_size * 2)
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size*2, 128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 64),            nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        out, _ = self.lstm(x)
        last   = self.layer_norm(out[:, -1, :])
        return self.classifier(self.dropout(last)).squeeze(1)


def train_lstm_fold(X_tr_in, y_tr, X_te_in, y_te,
                    n_epochs=N_EPOCHS, lr=LR, fold=0, label=''):
    scale_pw = torch.tensor(
        [(y_tr==0).sum() / (y_tr==1).sum()],
        dtype=torch.float32
    ).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=scale_pw)

    y_tr_t = torch.tensor(y_tr, dtype=torch.float32)
    y_te_t = torch.tensor(y_te, dtype=torch.float32)

    train_loader = DataLoader(
        TensorDataset(torch.tensor(X_tr_in), y_tr_t),
        batch_size=BATCH_SIZE, shuffle=True,
        num_workers=4, pin_memory=True
    )
    val_loader = DataLoader(
        TensorDataset(torch.tensor(X_te_in), y_te_t),
        batch_size=BATCH_SIZE, shuffle=False,
        num_workers=4, pin_memory=True
    )

    model = MortalityLSTM()
    if n_gpus > 1:
        model = nn.DataParallel(model)
    model = model.to(device)

    optimizer = torch.optim.Adam(
        model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', patience=3, factor=0.5)

    best_auprc = 0
    best_probs = None

    for epoch in range(1, n_epochs + 1):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        model.eval()
        probs_list = []
        with torch.no_grad():
            for xb, _ in val_loader:
                probs_list.append(
                    torch.sigmoid(model(xb.to(device))).cpu().numpy())
        probs = np.concatenate(probs_list)
        ap    = average_precision_score(y_te, probs)
        scheduler.step(ap)

        if ap > best_auprc:
            best_auprc = ap
            best_probs = probs.copy()

        if epoch % 10 == 0 or epoch == 1:
            print(f'    [{label}] fold {fold+1} ep {epoch:02d}  '
                  f'AUPRC={ap:.4f}{"  *" if ap==best_auprc else ""}')

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return best_probs, best_auprc


def normalise(X_input, X_train):
    ts_mean = X_train.mean(axis=(0,1), keepdims=True)
    ts_std  = X_train.std(axis=(0,1),  keepdims=True) + 1e-8
    return (X_input - ts_mean) / ts_std


def build_input(X_norm, M):
    return np.concatenate([X_norm, M], axis=2).astype(np.float32)


print('LSTM and training helpers defined.')

---
## 6. Run Variant A

For each fold:
1. Compute Black medians and White/Black observation rates from TRAINING data only
2. Apply Variant A imputation to TRAINING Black patients
3. Apply Variant A imputation to TEST Black patients (same rates from training fold)
4. Train LSTM from scratch on imputed training data
5. Evaluate on imputed test data

In [ ]:
rng_a = np.random.default_rng(42)
oof_variant_a = np.zeros(len(cohort))
fold_auprcs_a = []

print('Training Variant A — Black imputed to White observation rate')
print('=' * 65)

for fold, (train_idx, test_idx) in enumerate(fold_splits):
    print(f'\n── Fold {fold+1}/{N_FOLDS} ──────────────────────────────')

    # Compute fold statistics from training data only (no leakage)
    black_median, black_rate, white_rate = compute_fold_stats(train_idx)

    # Raw arrays for this fold
    X_tr = X_ts[train_idx].copy()
    M_tr = M_ts[train_idx].copy()
    X_te = X_ts[test_idx].copy()
    M_te = M_ts[test_idx].copy()
    y_tr = y[train_idx]
    y_te = y[test_idx]

    race_tr = race_arr[train_idx]
    race_te = race_arr[test_idx]

    # Apply Variant A to training and test Black patients
    X_tr_imp, M_tr_imp = apply_variant_a(
        X_tr, M_tr, race_tr, black_median, black_rate, white_rate, rng_a)
    X_te_imp, M_te_imp = apply_variant_a(
        X_te, M_te, race_te, black_median, black_rate, white_rate, rng_a)

    # Report how many slots were filled
    bl_tr = race_tr == 'Black'
    filled = (M_tr_imp[bl_tr] - M_tr[bl_tr]).sum()
    total_missing = (1 - M_tr[bl_tr]).sum()
    print(f'  Training: filled {filled:.0f} / {total_missing:.0f} '
          f'Black missing slots ({100*filled/total_missing:.1f}%)')

    # Normalise using IMPUTED training data statistics
    X_tr_n = normalise(X_tr_imp, X_tr_imp)
    X_te_n = normalise(X_te_imp, X_tr_imp)

    X_tr_in = build_input(X_tr_n, M_tr_imp)
    X_te_in = build_input(X_te_n, M_te_imp)

    # Train LSTM from scratch
    best_probs, best_auprc = train_lstm_fold(
        X_tr_in, y_tr, X_te_in, y_te,
        fold=fold, label='Variant A'
    )
    oof_variant_a[test_idx] = best_probs
    fold_auprcs_a.append(best_auprc)

    # Per-group results this fold
    for g in ['Black', 'White']:
        mask = race_te == g
        if mask.sum() > 0 and y_te[mask].sum() > 0:
            ap = average_precision_score(y_te[mask], best_probs[mask])
            print(f'  {g} AUPRC = {ap:.4f}')

np.save(IMP_DIR / 'oof_variant_a.npy', oof_variant_a)
print(f'\nVariant A complete!')
print(f'Fold AUPRCs: {[f"{x:.4f}" for x in fold_auprcs_a]}')
print(f'Mean: {np.mean(fold_auprcs_a):.4f} ± {np.std(fold_auprcs_a):.4f}')

---
## 7. Run Variant B

In [ ]:
oof_variant_b = np.zeros(len(cohort))
fold_auprcs_b = []

print('Training Variant B')
print('=' * 65)

for fold, (train_idx, test_idx) in enumerate(fold_splits):
    print(f'\n── Fold {fold+1}/{N_FOLDS} ──────────────────────────────')

    black_median, black_rate, white_rate = compute_fold_stats(train_idx)

    X_tr = X_ts[train_idx].copy()
    M_tr = M_ts[train_idx].copy()
    X_te = X_ts[test_idx].copy()
    M_te = M_ts[test_idx].copy()
    y_tr = y[train_idx]
    y_te = y[test_idx]

    race_tr = race_arr[train_idx]
    race_te = race_arr[test_idx]

    # Apply Variant B to training and test Black patients
    X_tr_imp, M_tr_imp = apply_variant_b(
        X_tr, M_tr, race_tr, black_median)
    X_te_imp, M_te_imp = apply_variant_b(
        X_te, M_te, race_te, black_median)

    # Report fill rate
    bl_tr = race_tr == 'Black'
    filled = (M_tr_imp[bl_tr] - M_tr[bl_tr]).sum()
    total_missing = (1 - M_tr[bl_tr]).sum()
    print(f'  Training: filled {filled:.0f} / {total_missing:.0f} '
          f'Black missing slots ({100*filled/total_missing:.1f}%)')

    X_tr_n  = normalise(X_tr_imp, X_tr_imp)
    X_te_n  = normalise(X_te_imp, X_tr_imp)
    X_tr_in = build_input(X_tr_n, M_tr_imp)
    X_te_in = build_input(X_te_n, M_te_imp)

    best_probs, best_auprc = train_lstm_fold(
        X_tr_in, y_tr, X_te_in, y_te,
        fold=fold, label='Variant B'
    )
    oof_variant_b[test_idx] = best_probs
    fold_auprcs_b.append(best_auprc)

    for g in ['Black', 'White']:
        mask = race_te == g
        if mask.sum() > 0 and y_te[mask].sum() > 0:
            ap = average_precision_score(y_te[mask], best_probs[mask])
            print(f'  {g} AUPRC = {ap:.4f}')

np.save(IMP_DIR / 'oof_variant_b.npy', oof_variant_b)
print(f'\nVariant B complete!')
print(f'Fold AUPRCs: {[f"{x:.4f}" for x in fold_auprcs_b]}')
print(f'Mean: {np.mean(fold_auprcs_b):.4f} ± {np.std(fold_auprcs_b):.4f}')

---
## 8. Results

In [ ]:
conditions = {
    'Baseline':  oof_baseline,
    'Variant A': oof_variant_a,
    'Variant B': oof_variant_b,
}

print('AUPRC by condition and race group:')
print('=' * 70)
print(f'{"Condition":<12}  {"Overall":>8}  {"Black":>8}  '
      f'{"White":>8}  {"Hispanic":>10}  {"Asian":>8}  {"Gap B-W":>8}')
print('-' * 70)

results = {}
for cond, probs in conditions.items():
    row = {}
    row['overall']  = average_precision_score(y, probs)
    for g in KEEP_RACES:
        mask = race_arr == g
        row[g] = average_precision_score(y[mask], probs[mask])
    row['gap'] = row['White'] - row['Black']
    results[cond] = row
    print(f'{cond:<12}  {row["overall"]:>8.4f}  {row["Black"]:>8.4f}  '
          f'{row["White"]:>8.4f}  {row["Hispanic"]:>10.4f}  '
          f'{row["Asian"]:>8.4f}  {row["gap"]:>+8.4f}')

print()
print('Change vs baseline:')
bl = results['Baseline']
for cond in ['Variant A', 'Variant B']:
    row = results[cond]
    print(f'  {cond}:')
    print(f'    Black delta:  {row["Black"]-bl["Black"]:+.4f}')
    print(f'    White delta:  {row["White"]-bl["White"]:+.4f}')
    print(f'    Gap delta:    {row["gap"]-bl["gap"]:+.4f} '
          f'({"narrowed" if row["gap"]<bl["gap"] else "widened"})')

---
## 9. Bootstrap CIs and p-values

In [ ]:
def bootstrap_auprc_ci(y_true, probs, n_boot=N_BOOTSTRAP, seed=42):
    rng = np.random.default_rng(seed)
    n   = len(y_true)
    scores = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        ys, ps = y_true[idx], probs[idx]
        if ys.sum() == 0 or ys.sum() == n: continue
        scores.append(average_precision_score(ys, ps))
    s = np.array(scores)
    return s.mean(), np.percentile(s, 2.5), np.percentile(s, 97.5)


def bootstrap_delta_p(y_true, p_base, p_new,
                       n_boot=N_BOOTSTRAP, seed=0):
    rng = np.random.default_rng(seed)
    n   = len(y_true)
    deltas = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        ys  = y_true[idx]
        if ys.sum() == 0 or ys.sum() == n: continue
        deltas.append(
            average_precision_score(ys, p_new[idx]) -
            average_precision_score(ys, p_base[idx])
        )
    d = np.array(deltas)
    p = 2 * min(np.mean(d <= 0), np.mean(d >= 0))
    return d.mean(), np.percentile(d, 2.5), np.percentile(d, 97.5), p


print('Bootstrap CIs and significance tests...')
print('=' * 65)

ci_records = []
for cond, probs in conditions.items():
    print(f'\n{cond}:')
    for g in KEEP_RACES:
        mask = race_arr == g
        mn, lo, hi = bootstrap_auprc_ci(y[mask], probs[mask])
        print(f'  {g:<12}  {mn:.4f}  [{lo:.4f}, {hi:.4f}]')
        ci_records.append({'condition': cond, 'group': g,
                           'auprc': mn, 'ci_lower': lo, 'ci_upper': hi})

print('\nSignificance of Black AUPRC change vs baseline:')
mask_b = race_arr == 'Black'
for cond in ['Variant A', 'Variant B']:
    mn, lo, hi, p = bootstrap_delta_p(
        y[mask_b], oof_baseline[mask_b], conditions[cond][mask_b])
    sig = '(significant)' if p < 0.05 else '(not significant)'
    print(f'  {cond:<12}  delta={mn:+.4f}  '
          f'[{lo:+.4f}, {hi:+.4f}]  p={p:.4f}  {sig}')

ci_df = pd.DataFrame(ci_records)
ci_df.to_csv(IMP_DIR / 'retrain_bootstrap_ci.csv', index=False)
print('\nSaved retrain_bootstrap_ci.csv')

---
## 10. Figures

In [ ]:
# Figure 1: Black and White AUPRC across three conditions
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

cond_names = list(conditions.keys())
x = np.arange(len(cond_names))

for ax, group, color in [
    (axes[0], 'Black', '#D65F5F'),
    (axes[1], 'White', '#4878CF'),
]:
    mask = race_arr == group
    vals = [average_precision_score(y[mask], conditions[c][mask])
            for c in cond_names]
    bars = ax.bar(cond_names, vals, color=[
        COLORS[c] for c in cond_names],
        edgecolor='white', width=0.5, alpha=0.85)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, val+0.003,
                f'{val:.4f}', ha='center', fontsize=10, fontweight='bold')
    ax.set_ylabel('AUPRC')
    ax.set_title(f'{group} patients — AUPRC by training condition')
    ax.set_ylim(0, max(vals)*1.15)

plt.suptitle('LSTM retrained on imputed data — Black and White AUPRC\n'
             'All imputation uses Black patient medians only', fontsize=12)
plt.tight_layout()
plt.savefig(IMP_DIR / 'fig_retrain_black_white.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig_retrain_black_white.png')

In [ ]:
# Figure 2: Disparity gap across conditions 
fig, ax = plt.subplots(figsize=(9, 5))

gaps   = [results[c]['gap'] for c in cond_names]
colors = [COLORS[c] for c in cond_names]
bars   = ax.bar(cond_names, gaps, color=colors,
                edgecolor='white', width=0.5, alpha=0.85)

for bar, gap in zip(bars, gaps):
    ax.text(bar.get_x()+bar.get_width()/2, gap+0.001,
            f'{gap:.4f}', ha='center', fontsize=11, fontweight='bold')

ax.set_ylabel('White AUPRC − Black AUPRC')
ax.set_title('Racial performance gap across training conditions\n'
             'Smaller = more equitable')
ax.set_ylim(0, max(gaps)*1.2)

# Add annotation arrows
for i, (c, gap) in enumerate(zip(cond_names[1:], gaps[1:]), 1):
    change = gap - gaps[0]
    ax.annotate(f'{change:+.4f} vs baseline',
                xy=(i, gap), xytext=(i, gap + 0.006),
                ha='center', fontsize=9,
                color='green' if change < 0 else 'red')

plt.tight_layout()
plt.savefig(IMP_DIR / 'fig_retrain_gap.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig_retrain_gap.png')

In [ ]:
# Figure 3: AUPRC by race group 
x     = np.arange(len(KEEP_RACES))
width = 0.25
fig, ax = plt.subplots(figsize=(12, 5))

for offset, (cond, probs) in enumerate(conditions.items()):
    vals = []
    for g in KEEP_RACES:
        mask = race_arr == g
        vals.append(average_precision_score(y[mask], probs[mask]))
    ax.bar(x + offset*width, vals, width, label=cond,
           color=COLORS[cond], edgecolor='white', alpha=0.85)

ax.set_xticks(x + width)
ax.set_xticklabels(KEEP_RACES)
ax.set_ylabel('AUPRC')
ax.set_title('AUPRC by race group — baseline vs imputation variants\n'
             'Only Black patients are imputed; White and others unchanged')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig(IMP_DIR / 'fig_retrain_by_race.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig_retrain_by_race.png')

In [ ]:
# Figure 4: PR curves for Black patients
fig, ax = plt.subplots(figsize=(8, 6))

mask_b = race_arr == 'Black'
yb     = y[mask_b]

ax.axhline(yb.mean(), color='gray', linestyle=':', alpha=0.6,
           label=f'Prevalence ({yb.mean():.2f})')

for cond, probs in conditions.items():
    pb = probs[mask_b]
    ap = average_precision_score(yb, pb)
    prec, rec, _ = precision_recall_curve(yb, pb)
    ax.plot(rec, prec, color=COLORS[cond], linewidth=2.5,
            label=f'{cond} (AUPRC={ap:.3f})')

ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('PR curves — Black patients only\nBaseline vs imputation variants')
ax.legend(fontsize=10)
ax.set_xlim(0,1); ax.set_ylim(0,1)
plt.tight_layout()
plt.savefig(IMP_DIR / 'fig_retrain_pr_black.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig_retrain_pr_black.png')

---
## 11. Summary

In [ ]:
print('=' * 70)
print('IMPUTATION RETRAINING SUMMARY')
print('=' * 70)
print(f'Cohort: {len(cohort):,}  |  Mortality: {y.mean():.1%}')
print(f'\nOnly Black patients were imputed using Black training medians.')
print(f'White, Hispanic, Asian patients were untouched in all conditions.')

print(f'\n{"Condition":<12}  {"Overall":>8}  {"Black":>8}  '
      f'{"White":>8}  {"Gap":>8}')
print('-' * 50)
for cond in cond_names:
    r = results[cond]
    print(f'{cond:<12}  {r["overall"]:>8.4f}  {r["Black"]:>8.4f}  '
          f'{r["White"]:>8.4f}  {r["gap"]:>+8.4f}')

print(f'\nKey question: did retraining on imputed data close the gap?')
gap_bl = results['Baseline']['gap']
gap_va = results['Variant A']['gap']
gap_vb = results['Variant B']['gap']

print(f'  Baseline gap:   {gap_bl:.4f}')
print(f'  Variant A gap:  {gap_va:.4f}  '
      f'({"narrowed" if gap_va < gap_bl else "widened"} by {abs(gap_va-gap_bl):.4f})')
print(f'  Variant B gap:  {gap_vb:.4f}  '
      f'({"narrowed" if gap_vb < gap_bl else "widened"} by {abs(gap_vb-gap_bl):.4f})')

print(f'\nOutputs saved to: {IMP_DIR}')
for f in sorted(IMP_DIR.glob('*')):
    print(f'  {f.name}')